# IS445 Homework: Bigfoot (BFRO) Sighting Reports with Altair
Two visualizations of the BFRO reports dataset, exported as Vega-Lite JSON for my Jekyll page.

In [1]:
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()

url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv'
df = pd.read_csv(url)
df.shape

(4747, 29)

## Data transformations
- Parse `date` into a timestamp and pull out the year (reports with no date are dropped for the time plot only).
- Keep only Class A and Class B reports (Class C is ~30 rows of second-hand accounts).
- Restrict the time plot to 1960–2018 so the line isn't dominated by a handful of very old reports and the dataset ends in 2018.
- For plot 2, aggregate to counts per state × season × classification so the exported JSON stays small.

In [2]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year
df = df[df['classification'].isin(['Class A', 'Class B'])]

yearly = (df.dropna(subset=['year'])
            .query('year >= 1960 and year <= 2018')
            .groupby(['year', 'classification']).size()
            .reset_index(name='reports'))
yearly['year'] = pd.to_datetime(yearly['year'].astype(int).astype(str), format='%Y')
yearly.head()

## Plot 1: Reports per year by classification

In [3]:
class_colors = alt.Scale(domain=['Class A', 'Class B'], range=['#1b9e77', '#7570b3'])

chart1 = alt.Chart(yearly).mark_line(point=True).encode(
    x=alt.X('year:T', title='Year of sighting'),
    y=alt.Y('reports:Q', title='Number of reports'),
    color=alt.Color('classification:N', scale=class_colors, title='Report class'),
    tooltip=[alt.Tooltip('year:T', format='%Y'), 'classification:N', 'reports:Q']
).properties(width='container', height=350,
             title='Bigfoot reports per year, 1960–2018')
chart1

In [4]:
chart1.save('bigfoot_by_year.json')

## Plot 2: Interactive seasons × states
Click a season bar (shift-click for several) to filter the state ranking; use the dropdown to pick a report class.

In [5]:
agg = (df[df['season'] != 'Unknown']
         .groupby(['state', 'season', 'classification']).size()
         .reset_index(name='reports'))

season_order = ['Spring', 'Summer', 'Fall', 'Winter']
season_colors = alt.Scale(domain=season_order,
                          range=['#66a61e', '#e6ab02', '#d95f02', '#1f78b4'])

season_sel = alt.selection_point(fields=['season'], on='click')
class_dd = alt.binding_select(options=[None, 'Class A', 'Class B'],
                              labels=['All', 'Class A', 'Class B'], name='Report class: ')
class_sel = alt.selection_point(fields=['classification'], bind=class_dd)

base = alt.Chart(agg).transform_filter(class_sel)

seasons = base.mark_bar().encode(
    x=alt.X('season:N', sort=season_order, title='Season'),
    y=alt.Y('sum(reports):Q', title='Number of reports'),
    color=alt.condition(season_sel,
                        alt.Color('season:N', scale=season_colors, legend=None),
                        alt.value('lightgray')),
    tooltip=['season:N', alt.Tooltip('sum(reports):Q', title='reports')]
).add_params(season_sel, class_sel).properties(width=220, height=400,
    title='Click a season')

states = base.transform_filter(season_sel).transform_aggregate(
    reports='sum(reports)', groupby=['state']
).transform_window(
    rank='rank(reports)', sort=[alt.SortField('reports', order='descending')]
).transform_filter('datum.rank <= 15').mark_bar(color='#555555').encode(
    x=alt.X('reports:Q', title='Number of reports'),
    y=alt.Y('state:N', sort='-x', title=None),
    tooltip=['state:N', 'reports:Q']
).properties(width=350, height=400, title='Top 15 states for the selection')

chart2 = (seasons | states).properties(
    title='Where and when Bigfoot is reported')
chart2

In [6]:
chart2.save('bigfoot_season_state.json')